# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

# SqLite 설정

In [17]:
import sqlite3

테이블 생성

In [18]:
DB = "prices.db"  # SQLite 데이터베이스 파일명 (같은 디렉토리에 자동 생성됨)

# with문: 작업 완료 후 DB 연결을 자동으로 닫아줌 (자원 누수 방지)
with sqlite3.connect(DB) as conn:  # prices.db 파일에 연결 (없으면 새로 생성)
    
    cursor = conn.cursor()  # SQL 쿼리를 실행할 커서 객체 생성
    
    cursor.execute(
        # CREATE TABLE IF NOT EXISTS: 테이블이 없을 때만 생성 (이미 있으면 오류 없이 넘어감)
        # prices: 테이블 이름
        # city TEXT PRIMARY KEY: 도시명(문자열), 기본키(중복 불가, NULL 불가)
        # price REAL: 가격(소수점 포함 실수)
        'CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)'
    )
    
    conn.commit()  # 변경사항을 DB 파일에 영구 저장 (commit 안 하면 반영 안 됨)

In [19]:
def set_ticket_price(city, price):
    # with문: 작업 완료 후 DB 연결을 자동으로 닫아줌
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()  # SQL 쿼리를 실행할 커서 객체 생성
        
        cursor.execute(
            # INSERT INTO prices: prices 테이블에 새 데이터 삽입
            # (city, price) VALUES (?, ?): ?는 SQL 인젝션 방지용 플레이스홀더
            # ON CONFLICT(city): city(기본키)가 중복될 경우 처리 방법 지정
            # DO UPDATE SET price = ?: 중복 시 삽입 대신 가격만 업데이트 (UPSERT)
            'INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?',
            (city.lower(), price, price)  # city는 소문자로 통일, price는 삽입/업데이트에 각각 사용
        )
        
        conn.commit()  # 변경사항을 DB 파일에 영구 저장

In [20]:
# 도시별 왕복 티켓 가격 딕셔너리 (도시명: 가격)
ticket_prices = {"런던": "$799", "파리": "$899", "도쿄": "$1400", "베를린": "$499"}

# 딕셔너리를 순회하며 각 도시의 가격을 DB에 저장
for city, price in ticket_prices.items():  # .items(): 키(city)와 값(price)을 동시에 추출
    set_ticket_price(city, price)           # DB에 INSERT 또는 UPDATE (UPSERT)

In [21]:
def get_ticket_price(city):
    # 툴이 실제로 호출되었는지 확인용 로그 출력
    # flush=True: 버퍼링 없이 즉시 출력 (실시간 로그 확인용)
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    
    # with문: 작업 완료 후 DB 연결을 자동으로 닫아줌
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()  # SQL 쿼리를 실행할 커서 객체 생성
        
        # SELECT price: price 컬럼만 조회
        # WHERE city = ?: city가 일치하는 행만 필터링
        # city.lower(): 대소문자 구분 없이 조회 (저장 시도 소문자로 통일했으므로)
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        
        result = cursor.fetchone()  # 조회 결과 1개 행을 튜플로 반환 (없으면 None)
        
        # result가 있으면 가격 반환, 없으면 데이터 없음 메시지 반환
        # result[0]: 튜플의 첫 번째 값 = price 컬럼 값
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [22]:
get_ticket_price("도쿄")


DATABASE TOOL CALLED: Getting price for 도쿄


'Ticket price to 도쿄 is $$1400'

# openAI 연동

In [8]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr


In [9]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()

DB = "prices.db"

OpenAI API Key exists and begins sk-proj-


In [10]:
system_message = """
당신은 FlightAI 항공사의 친절한 어시스턴트입니다.
짧고 정중한 답변을 제공하며, 한 문장을 넘기지 않습니다.
항상 정확한 정보를 제공하고, 모르는 경우 모른다고 말합니다.
"""

DB 조회 하는 함수

In [ ]:
def get_ticket_price(city):
    # 툴이 실제로 호출되었는지 확인용 로그 출력
    # flush=True: 버퍼링 없이 즉시 출력 (실시간 로그 확인용)
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    
    # with문: 작업 완료 후 DB 연결을 자동으로 닫아줌
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()  # SQL 쿼리를 실행할 커서 객체 생성
        
        # SELECT price: price 컬럼만 조회
        # WHERE city = ?: city가 일치하는 행만 필터링
        # city.lower(): 대소문자 구분 없이 조회 (저장 시도 소문자로 통일했으므로)
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        
        result = cursor.fetchone()  # 조회 결과 1개 행을 튜플로 반환 (없으면 None)
        
        # result가 있으면 가격 반환, 없으면 데이터 없음 메시지 반환
        # result[0]: 튜플의 첫 번째 값 = price 컬럼 값
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [12]:
get_ticket_price("파리")

DATABASE TOOL CALLED: Getting price for 파리


'Ticket price to 파리 is $$899'

In [23]:
# OpenAI Tool Calling에 사용할 함수 스펙 정의
price_function = {
    "name": "get_ticket_price",              # 함수 이름 (handle_tool_calls에서 호출 시 매칭)
    "description": "목적지 도시까지의 왕복 티켓 가격을 조회합니다.",  # 언제 이 함수를 쓸지 AI가 판단하는 근거
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",                          # 문자열 타입
                "description": "고객이 여행하고자 하는 목적지 도시",  # AI가 인자를 추출하는 근거
            },
        },
        "required": ["destination_city"],  # 필수 파라미터 지정
        "additionalProperties": False      # 정의되지 않은 추가 파라미터 허용 안 함
    }
}

# OpenAI API가 요구하는 tools 형식으로 감싸기
# type: "function" → 현재 OpenAI가 지원하는 유일한 툴 타입
# function: price_function → 위에서 정의한 함수 스펙
tools = [{"type": "function", "function": price_function}]

tools  # Jupyter에서 tools 구조 확인용 출력

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': '목적지 도시까지의 왕복 티켓 가격을 조회합니다.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': '고객이 여행하고자 하는 목적지 도시'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

In [24]:
def handle_tool_calls(message):
    responses = []  # 여러 툴 호출 결과를 담을 리스트

    for tool_call in message.tool_calls:  # OpenAI가 요청한 모든 툴 호출을 순회

        if tool_call.function.name == "get_ticket_price":  # 호출된 함수명이 가격 조회인지 확인
                                                            # (다른 함수가 추가될 경우를 대비한 분기)
            arguments = json.loads(tool_call.function.arguments)  # JSON 문자열 → 딕셔너리 변환
            city = arguments.get('destination_city')               # 목적지 도시명 추출
            price_details = get_ticket_price(city)                 # DB에서 실제 가격 조회

            responses.append({
                "role": "tool",                # OpenAI에게 툴 실행 결과임을 명시
                "content": price_details,      # DB에서 조회한 가격 정보
                "tool_call_id": tool_call.id   # 어떤 툴 호출에 대한 결과인지 OpenAI와 매핑
            })

    return responses  # 모든 툴 실행 결과 리스트 반환 → chat()에서 messages.extend()로 추가됨

In [25]:
def chat(message, history):
    # Gradio에서 전달된 대화 기록을 OpenAI 형식으로 변환
    history = [{"role":h["role"], "content":h["content"]} for h in history]

    # 시스템 메시지 + 대화 기록 + 새 사용자 메시지를 합쳐 전체 메시지 구성
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]

    # 1단계 요청: tools 포함하여 OpenAI에 전송
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    # 툴 호출이 필요한 동안 계속 반복 (연쇄 툴 호출 처리)
    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message      # OpenAI의 툴 호출 요청 메시지 추출
        responses = handle_tool_calls(message)      # 모든 툴 실행 후 결과 리스트 획득
        messages.append(message)                    # 툴 호출 요청을 대화 기록에 추가
        messages.extend(responses)                  # 툴 실행 결과들을 대화 기록에 개별 추가

        # tools 포함하여 재요청 (연쇄 툴 호출 가능하도록)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    # finish_reason == "stop" 이면 while 종료 → 최종 응답 텍스트 반환
    return response.choices[0].message.content

In [16]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Getting price for 런던
DATABASE TOOL CALLED: Getting price for 서울
